# Predicción de puntaje Saber 11 (Pipeline final)

**Aprendizaje de Máquina Aplicado** · Universidad EAFIT · 2026

**Equipo:** Patricia Arango, Santiago Higuita, Alexander Pelaez  
**Repositorio:** [ml_project_eafit](https://github.com/AlexanderPelaezJimenez/ml_project_eafit)

Este notebook es el **entregable final** del proyecto. Cierra el ciclo que arrancó con la entrega 1 (problema, datos, EDA y baseline) y muestra cómo, partiendo de las brechas socioeconómicas que vimos en el EDA, construimos un pipeline de regresión que predice `punt_global` y un champion serializado, listo para ser servido en una app.

## Contenido

1. [Resumen ejecutivo](#1-resumen-ejecutivo)
2. [Contexto y objetivos](#2-contexto-y-objetivos)
3. [Datos](#3-datos)
4. [Metodología](#4-metodología)
5. [Configuración global](#5-configuración-global)
6. [Carga del dataset y split](#6-carga-del-dataset-y-split)
7. [Pipeline de modelado](#7-pipeline-de-modelado)
8. [Tuning y selección del champion](#8-tuning-y-selección-del-champion)
9. [Evaluación en test holdout](#9-evaluación-en-test-holdout)
10. [Validaciones de robustez](#10-validaciones-de-robustez)
11. [Diagnósticos: residuales y fairness](#11-diagnósticos-residuales-y-fairness)
12. [Interpretabilidad con SHAP](#12-interpretabilidad-con-shap)
13. [Persistencia del champion](#13-persistencia-del-champion)
14. [Conclusiones](#14-conclusiones)

## 1. Resumen ejecutivo

[En construcción]

## 2. Contexto y objetivos

El examen Saber 11 (ICFES) es la prueba estandarizada que cierra la educación media en Colombia y condiciona el acceso a la universidad. En la entrega 1 confirmamos lo que la literatura ya sugería: hay brechas grandes y consistentes por estrato, educación de los padres, acceso a tecnología y tipo de colegio.

**Pregunta:** ¿qué tanto del puntaje global puede explicarse con variables socioeconómicas y del entorno educativo, sin usar puntajes parciales?

**Tarea:** regresión supervisada sobre `punt_global` (rango 0–500).

**Métrica primaria:** RMSE. Penaliza errores grandes, que en este dominio son los más costosos: un puntaje muy mal estimado afecta decisiones de admisión y de política. Reportamos también MAE y R² para facilitar la lectura.

**Uso previsto:** herramienta académica para investigación y análisis educativo.

## 3. Datos

Resultados Saber 11 del Portal de Datos Abiertos del Gobierno de Colombia. Después de la limpieza documentada en `03-data-cleaning.ipynb` trabajamos sobre `data/processed/cleaned_dataset.parquet`: 3.4M filas y 28 columnas, periodos 2014-2 a 2022-4.

Detalles (dominios, dtypes, decisiones de limpieza, sesgos conocidos): ver `docs/datacard.md`.

## 4. Metodología

### 4.1 Estrategia de validación y prevención de leakage

Tres líneas de defensa contra leakage:

1. **Eliminación de fuentes de leakage en cleaning.** En `03-data-cleaning` removimos los 5 puntajes parciales (`punt_matematicas`, `punt_lectura_critica`, etc.) y `desemp_ingles`. La correlación con `punt_global` era ≈0.99: incluirlos hubiera dado un R² sesgado sin aprender nada útil.

2. **Pipeline encapsulado.** Imputers, encoders y agregados van dentro de un `sklearn.Pipeline` que se reentrena por fold del CV únicamente con datos de train. Test holdout solo se transforma con el pipeline ya ajustado, nunca con `fit`.

3. **Group aggregates con leave-one-out.** El `GroupAggregatesTransformer` calcula `pct_internet_mcpio` y similares en `fit`. En `fit_transform` (fold de train) usamos LOO para que cada fila reciba el agregado calculado **sin su propia contribución**, eliminando el in-sample bias.

**Split principal:** estratificado por `periodo` (80/20). Es la pregunta razonable de negocio: ¿predigo bien dentro del rango de cohortes vistas? Como ese split deja municipios y colegios en train y test, complementamos con dos validaciones de robustez (sección 10):

- **`GroupKFold` por municipio**: ¿qué pasa con municipios no vistos?
- **Split temporal** (train ≤2019, test ≥2020): ¿qué pasa con cohortes futuras?

### 4.2 Modelos

Tres gradient boosting trees: **LightGBM, XGBoost y CatBoost**. Son robustos a multicolinealidad (relevante porque las features socioeconómicas están correlacionadas), manejan missing values nativamente y suelen ser el estado del arte en datos tabulares heterogéneos.

### 4.3 Tuning

Optuna con TPE (multivariado) + median pruner. Cada trial entrena el pipeline completo sobre los folds de CV; el pruner descarta trials malos temprano. Hiperparámetros buscados en `src/modeling/pipeline.py:PipelineML.build_model_grid`.

## 5. Configuración global

Toda la parametrización del experimento en una sola celda. Cambiarlos aquí afecta toda la ejecución del notebook.

In [1]:
# Configuración del experimento
SEED = 42
TEST_SIZE = 0.2
N_TRIALS = 40                       # trials de Optuna por modelo
CV_FOLDS = 3                        # folds del CV de tuning
USE_GPU = False                     # cambiar a True en máquina con GPU
MODELS_TO_TUNE = [
    'lightGBM',
    'xgboost',
    'catboost'
]

SHAP_SAMPLE_SIZE = 50_000           # muestra para SHAP
ROBUSTNESS_GROUP_FOLDS = 5          # folds del GroupKFold por municipio
ROBUSTNESS_TRAIN_UNTIL = '20194'    # split temporal: train hasta 2019-4

# Paths
from pathlib import Path

DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
FIGURES_DIR = Path('../reports/figures')
CHAMPION_DIR = MODELS_DIR / 'champion'

for d in (MODELS_DIR, FIGURES_DIR, CHAMPION_DIR):
    d.mkdir(parents=True, exist_ok=True)

### Imports

In [5]:
import os
import sys

sys.path.append(os.path.dirname(os.getcwd()))

In [7]:
import warnings

import numpy as np
import pandas as pd
import polars as pl
from loguru import logger
from sklearn.model_selection import train_test_split

# Pipeline y utilidades del proyecto
from src.modeling.constants import (
    CLIP_EDAD,
    MAPEO_CUARTOS,
    MAPEO_PERSONAS,
)
from src.modeling.pipeline import PipelineML
from src.modeling.preprocessor import build_preprocessor
from src.modeling.diagnostics import (
    calibration_by_bucket,
    fairness_report,
    plot_predicted_vs_true,
    plot_residual_distribution,
    plot_residual_vs_pred,
    residual_summary,
)
from src.modeling.validation import group_kfold_cv, temporal_holdout
from src.modeling.champion import save_champion

warnings.filterwarnings('ignore', message='X does not have valid feature names')

## 6. Carga del dataset y split

Cargamos el parquet limpio y aplicamos el split estratificado por periodo. El target sale como string del cleaning, lo casteamos a `Int32`.

In [8]:
cleaned_dataset = pl.scan_parquet(DATA_DIR / 'cleaned_dataset.parquet').collect()
logger.info(f'Dimensiones del dataset: {cleaned_dataset.shape}')
cleaned_dataset.head(5)

2026-05-23 21:54:56.810 | INFO     | __main__:<module>:2 - Dimensiones del dataset: (3395834, 28)


cole_sede_principal,cole_caracter,fami_cuartoshogar,fami_educacionpadre,estu_tipodocumento,estu_fechanacimiento,fami_tienecomputador,estu_genero,fami_tienelavadora,fami_personashogar,estu_depto_presentacion,estu_depto_reside,cole_naturaleza,cole_calendario,estu_mcpio_reside,periodo,cole_genero,fami_estratovivienda,cole_area_ubicacion,cole_bilingue,cole_mcpio_ubicacion,estu_mcpio_presentacion,fami_educacionmadre,fami_tieneautomovil,cole_jornada,punt_global,fami_tieneinternet,cole_depto_ubicacion
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""N""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria completa""","""TI""","""15/02/2003""","""Si""","""F""","""Si""","""1 a 2""","""HUILA""","""HUILA""","""OFICIAL""","""A""","""AIPE""","""20194""","""MIXTO""","""Estrato 2""","""RURAL""","""N""","""AIPE""","""AIPE""","""Postgrado""","""No""","""COMPLETA""","""339""","""Si""","""HUILA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria incompleta""","""TI""","""08/03/2002""","""No""","""F""","""No""","""5 a 6""","""HUILA""","""HUILA""","""OFICIAL""","""A""","""LA PLATA""","""20194""","""MIXTO""","""Estrato 1""","""URBANO""","""N""","""LA PLATA""","""LA PLATA""","""Primaria incompleta""","""No""","""COMPLETA""","""199""","""No""","""HUILA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Cuatro""","""Secundaria (Bachillerato) comp…","""TI""","""27/07/2000""","""Si""","""M""","""Si""","""Cinco""","""VALLE""","""VALLE""","""OFICIAL""","""A""","""CALI""","""20162""","""MIXTO""","""Estrato 5""","""URBANO""","""N""","""CALI""","""CALI""","""Secundaria (Bachillerato) inco…","""Si""","""MAÑANA""","""272""","""Si""","""VALLE"""
"""S""","""TÉCNICO/ACADÉMICO""","""Cinco""","""Primaria incompleta""","""TI""","""07/12/1999""","""No""","""F""","""Si""","""5 a 6""","""ANTIOQUIA""","""ANTIOQUIA""","""NO OFICIAL""","""A""","""MEDELLÍN""","""20172""","""MIXTO""","""Estrato 2""","""URBANO""","""N""","""MEDELLIN""","""ITAGÜÍ""","""Secundaria (Bachillerato) inco…","""No""","""SABATINA""","""253""","""Si""","""ANTIOQUIA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria incompleta""","""CC""","""15/09/1995""","""No""","""M""","""No""","""Cuatro""","""TOLIMA""","""TOLIMA""","""OFICIAL""","""A""","""SAN LUIS""","""20142""","""MIXTO""","""Estrato 1""","""RURAL""","""N""","""SAN LUIS""","""GUAMO""","""Primaria incompleta""","""No""","""MAÑANA""","""212""","""No""","""TOLIMA"""


In [9]:
X = cleaned_dataset.drop('punt_global')
y = cleaned_dataset['punt_global']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=cleaned_dataset['periodo'],
)

# Conversión a pandas/numpy: el pipeline de sklearn trabaja con pandas.
X_train_pd = X_train.to_pandas()
X_test_pd = X_test.to_pandas()
y_train_np = y_train.cast(pl.Int32).to_numpy().astype(np.float32)
y_test_np = y_test.cast(pl.Int32).to_numpy().astype(np.float32)

logger.info(f'Train: {X_train_pd.shape[0]:,} filas | Test: {X_test_pd.shape[0]:,} filas')
logger.info(f'Proporción test: {X_test_pd.shape[0] / cleaned_dataset.shape[0] * 100:.1f}%')

2026-05-23 21:55:07.491 | INFO     | __main__:<module>:18 - Train: 2,716,667 filas | Test: 679,167 filas
2026-05-23 21:55:07.491 | INFO     | __main__:<module>:19 - Proporción test: 20.0%


## 7. Pipeline de modelado

El pipeline tiene cuatro pasos, en este orden, y se reentrena entero en cada fold:

1. **`DeterministicFeatureProcessor`**: feature engineering puramente determinista (capital tecnológico, hacinamiento, brechas educativas, índice socioeconómico, edad aproximada). Stateless: no aprende nada ni genera estadísticas.

2. **`GroupAggregatesTransformer`**: agregados por municipio y departamento, con leave-one-out en train para evitar in-sample bias. Aquí no usamos el target.

3. **`ColumnTransformer`**: imputación + encoding (ordinal con orden explícito para estrato y educación; OHE para nominales de baja cardinalidad).

4. **Modelo**: LightGBM, XGBoost o CatBoost.

Los transformers viven en `src/modeling/transformers.py` y están cubiertos por tests unitarios en `tests/test_transformers.py`.

In [10]:
preprocessor = build_preprocessor()

dev_pipeline_ml = PipelineML(
    preprocessor=preprocessor,
    mapeo_personas=MAPEO_PERSONAS,
    mapeo_cuartos=MAPEO_CUARTOS,
    clip_edad=CLIP_EDAD,
    n_trials_tree=N_TRIALS,
    cv_folds=CV_FOLDS,
    use_gpu=USE_GPU,
    random_seed=SEED,
)

2026-05-23 21:55:14.329 | INFO     | src.modeling.pipeline:__init__:98 - Maquina detectada: 10 cores. Modo: CPU. parallel_trials=2, cores_per_model=5, concurrencia total=10


## 8. Tuning y selección del champion

Optuna optimiza RMSE de CV para cada uno de los tres modelos. La elección del champion la hacemos manualmente sobre la tabla comparativa: si dos modelos están empatados en RMSE, elegiremos el más simple (menos depth, menos n_estimators) por mantenibilidad.

In [11]:
studies = dev_pipeline_ml.tune(X_train_pd, y_train_np, models=MODELS_TO_TUNE)
comparison = dev_pipeline_ml.comparison_table()

comparison

2026-05-23 21:55:42.177 | INFO     | src.modeling.pipeline:tune:301 - Ejecutando modelo lightGBM...
/Users/santiago.higuitau/Documents/Machine_Learning/ml_project_eafit/src/modeling/pipeline.py:305: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(
/Users/santiago.higuitau/Documents/Machine_Learning/ml_project_eafit/src/modeling/pipeline.py:305: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(
[I 2026-05-23 21:55:42,179] A new study created in memory with name: Experimento usando: lightGBM


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-05-23 21:58:27,914] Trial 0 finished with value: 37.70796115667616 and parameters: {'n_estimators': 704, 'max_depth': 5, 'learning_rate': 0.02633615548025805, 'num_leaves': 140, 'min_child_samples': 45, 'subsample': 0.7870589481336246, 'colsample_bytree': 0.7043976266546974, 'reg_alpha': 0.6399703505865966, 'reg_lambda': 1.9432063936611448e-07}. Best is trial 0 with value: 37.70796115667616.
[I 2026-05-23 21:59:25,709] Trial 1 finished with value: 36.908466490741596 and parameters: {'n_estimators': 901, 'max_depth': 6, 'learning_rate': 0.2920153492714406, 'num_leaves': 119, 'min_child_samples': 15, 'subsample': 0.8975110064929662, 'colsample_bytree': 0.6802640110320414, 'reg_alpha': 1.5108902848378722, 'reg_lambda': 5.871584698751762e-05}. Best is trial 1 with value: 36.908466490741596.
[I 2026-05-23 22:01:15,265] Trial 2 finished with value: 37.18958495157238 and parameters: {'n_estimators': 558, 'max_depth': 4, 'learning_rate': 0.26847730262115427, 'num_leaves': 112, 'min_chi

2026-05-23 23:23:03.130 | INFO     | src.modeling.pipeline:tune:324 - lightGBM | Mejor RMSE CV: 36.7565
2026-05-23 23:23:03.131 | INFO     | src.modeling.pipeline:tune:325 - lightGBM | Mejores params: {'n_estimators': 980, 'max_depth': 10, 'learning_rate': 0.10410650734470608, 'num_leaves': 140, 'min_child_samples': 51, 'subsample': 0.6907705349700841, 'colsample_bytree': 0.6849563477166827, 'reg_alpha': 1.8559515428203663e-07, 'reg_lambda': 0.0008541815215158681}
2026-05-23 23:23:03.131 | INFO     | src.modeling.pipeline:tune:301 - Ejecutando modelo xgboost...
/Users/santiago.higuitau/Documents/Machine_Learning/ml_project_eafit/src/modeling/pipeline.py:305: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(
/Users/santiago.higuitau/Documents/Machine_Learning/ml_project_eafit/src/modeling/pipeline.py:305: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in t

[I 2026-05-23 23:23:03,125] Trial 39 finished with value: 36.76891492437115 and parameters: {'n_estimators': 858, 'max_depth': 9, 'learning_rate': 0.11125654079811283, 'num_leaves': 149, 'min_child_samples': 88, 'subsample': 0.6676251766860445, 'colsample_bytree': 0.8073679621088072, 'reg_alpha': 1.0986889667401257e-06, 'reg_lambda': 0.0059360117403817825}. Best is trial 29 with value: 36.75650385628597.


...

2026-05-24 02:43:59.835 | INFO     | src.modeling.pipeline:tune:324 - xgboost | Mejor RMSE CV: 36.7155
2026-05-24 02:43:59.838 | INFO     | src.modeling.pipeline:tune:325 - xgboost | Mejores params: {'n_estimators': 924, 'max_depth': 11, 'learning_rate': 0.0374981840353519, 'subsample': 0.8292889810829691, 'colsample_bytree': 0.6451099008936642, 'reg_alpha': 4.717356090090296, 'reg_lambda': 4.801400769311855, 'min_child_weight': 44, 'gamma': 4.281698670929322}
2026-05-24 02:43:59.838 | INFO     | src.modeling.pipeline:tune:301 - Ejecutando modelo catboost...
/Users/santiago.higuitau/Documents/Machine_Learning/ml_project_eafit/src/modeling/pipeline.py:305: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(
/Users/santiago.higuitau/Documents/Machine_Learning/ml_project_eafit/src/modeling/pipeline.py:305: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the 

[I 2026-05-24 02:43:59,819] Trial 39 finished with value: 36.75683085123698 and parameters: {'n_estimators': 916, 'max_depth': 10, 'learning_rate': 0.09928319044333545, 'subsample': 0.899417254577213, 'colsample_bytree': 0.6147208545506473, 'reg_alpha': 6.156992063825704, 'reg_lambda': 3.0262500643785275, 'min_child_weight': 47, 'gamma': 0.46097795086402227}. Best is trial 37 with value: 36.7155392964681.


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-05-24 02:49:59,348] Trial 1 finished with value: 36.9034352762832 and parameters: {'iterations': 597, 'depth': 8, 'learning_rate': 0.13912338141551545, 'l2_leaf_reg': 0.009408295261284108, 'bagging_temperature': 0.22814143140740184, 'random_strength': 0.015124913751696839}. Best is trial 1 with value: 36.9034352762832.
[I 2026-05-24 02:52:14,572] Trial 0 finished with value: 36.934832678151714 and parameters: {'iterations': 355, 'depth': 10, 'learning_rate': 0.12172114771664605, 'l2_leaf_reg': 0.3278769281461089, 'bagging_temperature': 0.6108239228790048, 'random_strength': 3.4193072792842236}. Best is trial 1 with value: 36.9034352762832.
[I 2026-05-24 02:53:31,484] Trial 2 finished with value: 37.09140965634172 and parameters: {'iterations': 247, 'depth': 8, 'learning_rate': 0.19810233088089554, 'l2_leaf_reg': 0.00439527471356476, 'bagging_temperature': 0.2086675598282005, 'random_strength': 0.001194876897857342}. Best is trial 1 with value: 36.9034352762832.
[I 2026-05-24 02

: 

In [ ]:
# Elegimos el champion del top de la tabla. Ajustar manualmente si hay empate técnico.
CHAMPION = comparison.iloc[0]['model']
logger.info(f'Champion seleccionado: {CHAMPION}')

dev_pipeline_ml.fit_best(CHAMPION, X_train_pd, y_train_np)

## 9. Evaluación en test holdout

Llamamos `evaluate` **una sola vez** sobre el test que reservamos al principio. Estas son las métricas oficiales del modelo.

In [ ]:
test_metrics = dev_pipeline_ml.evaluate(X_test_pd, y_test_np)
test_metrics

## 10. Validaciones de robustez

El test holdout responde "¿qué tan bien predice en cohortes vistas?". Las dos validaciones siguientes responden preguntas distintas y completan el panorama del modelo:

- **GroupKFold por municipio:** simula predecir en municipios no vistos. Si el RMSE sube mucho aquí, el modelo dependía de "memorizar" particularidades del municipio.

- **Split temporal:** entrena con periodos hasta 2019-4 y evalúa con 2020+. Detecta concept drift (cambios en composición demográfica, pandemia, ajustes del ICFES).

In [ ]:
# Reusamos los hiperparámetros del champion (no re-tuneamos).
best_params = dict(dev_pipeline_ml.studies[CHAMPION].best_params)

def build_champion():
    return dev_pipeline_ml.build_pipeline(
        model_selected=CHAMPION,
        params=best_params,
        model_type='tree',
    )

group_cv = group_kfold_cv(
    build_pipeline_fn=build_champion,
    X=X_train_pd,
    y=y_train_np,
    group_col='estu_mcpio_reside',
    n_splits=ROBUSTNESS_GROUP_FOLDS,
)
{k: v for k, v in group_cv.items() if k != 'folds'}

In [ ]:
# Split temporal sobre el dataset completo (para tener periodos pasados y futuros).
X_all_pd = cleaned_dataset.drop('punt_global').to_pandas()
y_all_np = cleaned_dataset['punt_global'].cast(pl.Int32).to_numpy().astype(np.float32)

temporal = temporal_holdout(
    build_pipeline_fn=build_champion,
    X=X_all_pd,
    y=y_all_np,
    period_col='periodo',
    train_until=ROBUSTNESS_TRAIN_UNTIL,
)
{k: v for k, v in temporal.items() if k not in ('train_periods', 'test_periods')}

In [ ]:
# Tabla resumen: tres esquemas de evaluación, una sola lectura.
robustness_summary = pd.DataFrame([
    {
        'scheme': 'Test holdout (random, stratified by periodo)',
        'rmse': test_metrics['rmse'],
        'mae': test_metrics['mae'],
        'r2': test_metrics['r2'],
        'n': test_metrics['n_test']
    },
    {
        'scheme': f"GroupKFold por municipio (n_splits={ROBUSTNESS_GROUP_FOLDS})",
        'rmse': group_cv['rmse_mean'],
        'mae': group_cv['mae_mean'],
        'r2': group_cv['r2_mean'],
        'n': len(X_train_pd)},
    {
        'scheme': f"Temporal (train<={ROBUSTNESS_TRAIN_UNTIL})",
        'rmse': temporal['rmse'],
        'mae': temporal['mae'],
        'r2': temporal['r2'],
        'n': temporal['n_test']},
])

robustness_summary

## 11. Diagnósticos: residuales y fairness

Métricas globales pueden esconder mal comportamiento sistemático. Aquí miramos los residuales y desagregamos las métricas por subgrupos sociodemográficos. Esta sección creemos que **NO** es opcional, sobre todo para nuestro dominio, pues publicar un modelo que prediga peor a estudiantes de estrato bajo sería exactamente lo contrario de lo que queremos y esperamos.

In [ ]:
y_pred = dev_pipeline_ml.best_pipeline.predict(X_test_pd)
summary = residual_summary(y_test_np, y_pred)
summary

In [ ]:
plot_predicted_vs_true(y_test_np, y_pred, save_path=FIGURES_DIR / 'pred_vs_true.png')

In [ ]:
plot_residual_distribution(y_test_np, y_pred, save_path=FIGURES_DIR / 'residual_distribution.png')

In [ ]:
plot_residual_vs_pred(y_test_np, y_pred, save_path=FIGURES_DIR / 'residual_vs_pred.png')

In [ ]:
calibration_by_bucket(y_test_np, y_pred, n_buckets=10)

### Fairness por subgrupos

Reportamos RMSE/MAE/R² y bias (sesgo medio) por estrato, género, área (urbano/rural), naturaleza del colegio (oficial/privado) y departamento. Diferencias grandes en RMSE entre subgrupos indican que el modelo aprendió patrones desiguales.

In [ ]:
fairness = fairness_report(y_test_np, y_pred, X_test_pd)

for col, df in fairness.items():
    print(f'\n--- {col} ---')
    display(df)

## 12. Interpretabilidad con SHAP

SHAP sobre una muestra de **train**, no de test (calcular SHAP sobre test para tomar decisiones de feature selection sería leakage). Buscamos:

- Que las top features coincidan con los hallazgos del EDA: estrato, educación de padres, capital tecnológico.

- Monotonicidad razonable en variables ordinales (a más estrato, más puntaje) en los dependence plots.

- Que ninguna feature derivada "trivialice" la señal (señal de que es un near-duplicate del target).

In [ ]:
importance_df = dev_pipeline_ml.explain(X_train_pd, sample_size=SHAP_SAMPLE_SIZE)
importance_df.head(25)

In [ ]:
dev_pipeline_ml.plot_shap_summary(save_path=FIGURES_DIR / 'shap_summary.png')

In [ ]:
dev_pipeline_ml.plot_shap_importance(save_path=FIGURES_DIR / 'shap_importance.png')

## 13. Persistencia del champion

Guardamos un bundle completo en `models/champion/`: pipeline serializado, metadata (versión de paquetes, semilla, hash del dataset), métricas de los tres esquemas, hiperparámetros, schema de features y model card. Esto es lo que la app de Streamlit va a cargar.

In [ ]:
metrics_bundle = {
    'test_holdout': test_metrics,
    'group_kfold': group_cv,
    'temporal': temporal,
}

save_champion(
    pipeline=dev_pipeline_ml.best_pipeline,
    out_dir=CHAMPION_DIR,
    model_name=CHAMPION,
    best_params=best_params,
    metrics=metrics_bundle,
    shap_importance=importance_df,
    X_train_sample=X_train_pd.head(1_000),
    target_col='punt_global',
    data_path=DATA_DIR / 'cleaned_dataset.parquet',
    seed=SEED,
    extra_metadata={
        'authors': ['Patricia Arango', 'Santiago Higuita', 'Alexander Pelaez'],
        'course': 'Aprendizaje de Máquina Aplicado',
        'institution': 'Universidad EAFIT',
    },
)

## 14. Conclusiones

[En construcción]

- Champion, métricas, comparación contra baseline (LinearRegression / DecisionTree de la entrega 1).
- Lectura de la tabla de robustez: ¿el modelo aguanta el cambio a municipios o periodos no vistos?
- Lectura del fairness report: ¿hay subgrupos donde el modelo se comporta peor?
- Top features SHAP y conexión con los gap del EDA.

### Limitaciones

- Sesgo de selección: solo estudiantes que presentaron Saber 11.

- Concept drift: datos hasta 2022. Cambios futuros (pandemia, ajustes ICFES) degradan el modelo.

- Variables auto-reportadas: estrato, internet, educación de padres pueden tener sesgo de respuesta.

- **Uso ético:** El uso legítimo del modelo es para análisis académico.